# ⚡ AegisX — Train on Colab (free T4)

Trains AegisX-Mini from scratch (or fine-tunes it later) and pushes the
weights straight to Hugging Face Hub.

**Your laptop never trains anything.** Run this notebook in Colab:
`File > Upload notebook`, then `Runtime > Run all`.

## 1. Setup

In [ ]:
!pip install -q torch huggingface_hub

import os
import sys
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Optional: mount Google Drive so checkpoints survive session disconnects.
# Set USE_DRIVE = False to skip the Google login and save locally instead.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount — checkpoints will be lost if the session disconnects.')

## 2. Get the AegisX code

Clone the repo (or upload your local copy with `aegisx/`, `data/`, `targets/`).

In [ ]:
WORK = '/content/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)

# Option A: clone from GitHub
!git clone https://github.com/FerzDevZ/AegisX.git .

# Option B: upload your local files
from google.colab import files
print('If your aegisx/ and data/ are not here yet, upload them or use Option A.')
print(os.listdir(WORK))

## 3. Configure training

Default: **from scratch** on the bundled seed corpus. Drop bigger `.txt`
corpora into `data/raw/` for a smarter model.

In [ ]:
# --- training hyperparameters ---
DATA_DIR      = 'data/raw'
OUT_DIR       = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-mini' if USE_DRIVE else '/content/aegisx/checkpoints/aegisx-mini'  # survives disconnects when Drive is mounted
VOCAB_SIZE    = 4096
BLOCK_SIZE    = 256
N_LAYER       = 6
N_HEAD        = 6
N_EMBD        = 384
BATCH_SIZE    = 16
GRAD_ACCUM    = 4
MAX_STEPS     = 5000
LR            = 3e-4
WARMUP_STEPS  = 200
EVAL_EVERY    = 500
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

## 4. Train

In [ ]:
!python -m aegisx.train \
    --data {DATA_DIR} \
    --out {OUT_DIR} \
    --vocab-size {VOCAB_SIZE} \
    --block-size {BLOCK_SIZE} \
    --n-layer {N_LAYER} \
    --n-head {N_HEAD} \
    --n-embd {N_EMBD} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --max-steps {MAX_STEPS} \
    --lr {LR} \
    --warmup-steps {WARMUP_STEPS} \
    --eval-every {EVAL_EVERY} \
    --device {DEVICE}

## 5. Quick sanity check

In [ ]:
!python -m aegisx.chat --model {OUT_DIR}/model.pt --tokenizer {OUT_DIR}/tokenizer.json \
    --prompt "You are AegisX, a cybersecurity assistant. User: how do I enumerate subdomains?\n\nAegisX:" \
    --max-new-tokens 150 --temperature 0.8 --top-k 50

## 6. Push to Hugging Face Hub

In [ ]:
from huggingface_hub import login, HfApi

# Creates a write token at https://huggingface.co/settings/tokens
HF_TOKEN = input('Paste your Hugging Face write token: ')
REPO_ID  = input('Repo id, e.g. youruser/aegisx-mini: ')
login(token=HF_TOKEN, add_to_git_credential=True)

In [ ]:
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True, private=False)
api.upload_folder(
    repo_id=REPO_ID,
    folder_path=OUT_DIR,
    repo_type='model',
    commit_message='AegisX-Mini from-scratch checkpoint',
)
print('Pushed to https://huggingface.co/' + REPO_ID)

## 7. Auto-deploy the chat UI to a free Hugging Face Space

Creates a Gradio Space, uploads the chat app + the `aegisx/` package, and sets
the `AEGISX_REPO` secret so the Space loads your model from the Hub.
All free (CPU basic).

In [ ]:
SPACE_ID = input('Space id, e.g. youruser/aegisx-mini-space: ')
api.create_repo(SPACE_ID, repo_type='space', space_sdk='gradio', exist_ok=True)
api.upload_file(repo_id=SPACE_ID, repo_type='space', path_in_repo='app.py', path_or_fileobj='hf/space_app.py')
api.upload_file(repo_id=SPACE_ID, repo_type='space', path_in_repo='requirements.txt', path_or_fileobj='hf/requirements.txt')
api.upload_folder(repo_id=SPACE_ID, repo_type='space', folder_path='aegisx', path_in_repo='aegisx')
api.add_space_secret(SPACE_ID, 'AEGISX_REPO', REPO_ID)
print('Space created: https://huggingface.co/spaces/' + SPACE_ID)